In [2]:
# IMPORTS

import pandas as pd

In [20]:
# LOAD & INSPECT DATASET

# Load train & test datas
train_set = pd.read_csv('../data/train.csv')
test_set = pd.read_csv('../data/test.csv')

# First look at training set
print(train_set.shape)
print(train_set.dtypes)

# 699 635 training rows
# 22 features + 1 target (satisfaction (boolean))


(699635, 23)
id                                     int64
Age                                    int64
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
Gender                                object
Customer Type                         object
Type of Travel                        object
Class                                 obje

In [34]:
# HANDLING MISSING VALUES

# Count NAs per column in training set
print(train_set.isna().sum())

# Count NAs per column in test set
print(test_set.isna().sum())


# 'Arrival Delay in Minutes' has NAs, look into whether 'Departure Delay in Minutes' can help fill those
print((train_set['Arrival Delay in Minutes'] - train_set['Departure Delay in Minutes']).describe())
diff = train_set['Arrival Delay in Minutes'] - train_set['Departure Delay in Minutes']
print(f"On {(diff.abs() == 0).mean().round(4) * 100}% of rows, arrival and departure delay match exactly.")
print(f"On {(diff.abs() <= 5).mean().round(4) * 100}% of rows, arrival and departure delay differ by at most 5 minutes.")
print(f"On {(diff.abs() <= 10).mean().round(4) * 100}% of rows, arrival and departure delay differ by at most 10 minutes.")

# We assume we can use departure delay as a proxy for arrival delay in the NA cases
train_set['Arrival Delay in Minutes'] = train_set['Arrival Delay in Minutes'].fillna(train_set['Departure Delay in Minutes'])
test_set['Arrival Delay in Minutes'] = test_set['Arrival Delay in Minutes'].fillna(test_set['Departure Delay in Minutes'])

# Count NAs per column
print(f"\n{train_set['Arrival Delay in Minutes'].isna().sum()} remaining NAs in training set.")
print(f"\n{test_set['Arrival Delay in Minutes'].isna().sum()} remaining NAs in test set.")



# Service rating columns should not contain any values outside 1-5, we count the instances where they contain 0s (Not rated)
service_cols = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
print((train_set[service_cols] == 0).sum().sort_values(ascending=False))

# As there is too many, we create an extra dummy column for each service rating column indicating if it is rated or not

# For train and test set...
for df in (train_set, test_set):

    # For each service rating column...
    for column in service_cols:
        
        # Set a binary dummy of whether the service rating column is rated...
        df[f'{column} rated'] = (df[column] != 0).astype(int)

        # ...and set 0s to NA in original columns 
        df[column] = df[column].where(df[column] != 0)

Age                                  0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
Ease of Online booking               0
Gate location                        0
Food and drink                       0
Online boarding                      0
Seat comfort                         0
Inflight entertainment               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Cleanliness                          0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
Gender                               0
Customer Type                        0
Type of Travel                       0
Class                                0
satisfaction                         0
dtype: int64
id                                   0
Age                                  0
Flight Distance                      0
Inflight wif

In [22]:
# OUTLIER CHECK

# Check whether any feature has unreasonable min/max values 
print(train_set.describe(percentiles=[0.01, 0.99]).T)

# Drop the ID column, which is obviously not informative
train_set = train_set.drop(columns='id')


                                      count           mean            std  \
id                                 699635.0  349817.000000  201967.372130   
Age                                699635.0      39.001928      13.800943   
Flight Distance                    699635.0    1353.778702     954.422566   
Inflight wifi service              699635.0       2.778722       1.280419   
Departure/Arrival time convenient  699635.0       3.156210       1.475390   
Ease of Online booking             699635.0       2.813240       1.329550   
Gate location                      699635.0       3.013734       1.254014   
Food and drink                     699635.0       3.265644       1.300202   
Online boarding                    699635.0       3.364156       1.288538   
Seat comfort                       699635.0       3.568950       1.275333   
Inflight entertainment             699635.0       3.453910       1.308157   
On-board service                   699635.0       3.517404       1.238474   

In [39]:
# CHECK FOR CLASS IMBALANCE IN PREDICTORS & TARGET
# (relatively even distirbution --> no issue to handle)

# For categorical columns...
for column in train_set.select_dtypes('object'):

    # Count proportion per category
    print(train_set[column].value_counts(normalize=True), '\n')

print(train_set['satisfaction'].value_counts(normalize=True))



Gender
Male      0.502666
Female    0.497334
Name: proportion, dtype: float64 

Customer Type
Loyal Customer       0.824701
disloyal Customer    0.175299
Name: proportion, dtype: float64 

Type of Travel
Business travel    0.711001
Personal Travel    0.288999
Name: proportion, dtype: float64 

Class
Business    0.489129
Eco         0.467964
Eco Plus    0.042907
Name: proportion, dtype: float64 

satisfaction
0    0.556427
1    0.443573
Name: proportion, dtype: float64


In [40]:
# MAP CATEGORICAL VARIABLES INTO NUMERICAL FORMAT

# Map binary categorical variables into binary format
for df in (train_set, test_set):
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
    df['Customer Type'] = df['Customer Type'].map({'disloyal Customer': 0, 'Loyal Customer': 1})
    df['Type of Travel'] = df['Type of Travel'].map({'Personal Travel': 0, 'Business travel': 1})


# Map non-binary, but ordinal, categorical variables into numerical format
for df in (train_set, test_set):    
    df['Class'] = df['Class'].map({'Eco': 0, 'Eco Plus': 1, 'Business': 2})


In [42]:
# CONSTRUCT TRAINING SET

# Predictors
X = train_set.drop(columns='satisfaction')

# Target
y = train_set['satisfaction']

# Split data into 5 stratified folds 
# (setup for hyperparameter tuning)
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

(699635, 34) (699635,)


In [51]:
# TRAIN XGBOOST CLASSIFIER

from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
import time
from sklearn.model_selection import ParameterGrid


# Train a default model as baseline
scores = cross_val_score(XGBClassifier(), X, y, cv=cv, scoring='roc_auc')
print(f"XGBoost with default settings, 5-fold CV ROC AUC: mean {scores.mean():.4f}, std {scores.std():.4f}")

# Parameter grid to grid search through
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.03, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 5],
}


results_xgb = []

# Define all hyperparameter combinations
combos = list(ParameterGrid(param_grid))

# For each combination...
for i, params in enumerate(combos, 1):

    start = time.time()

    # Train the model using 5-fold CV, scored by ROC AUC
    scores = cross_val_score(XGBClassifier(**params), X, y, cv=cv, scoring='roc_auc')

    # Save results (mean + std)
    results_xgb.append({**params, 'mean_auc': scores.mean(), 'std_auc': scores.std()})
    print(f"XGBoost grid search, combination {i}/{len(combos)}: {params} -> 5-fold CV ROC AUC mean {scores.mean():.4f}, std {scores.std():.4f}, time {time.time() - start:.0f}s")

# Save results to a df & print top 10 
results_xgb = pd.DataFrame(results_xgb).sort_values('mean_auc', ascending=False)
print("\nXGBoost grid search, top 10 combinations by 5-fold CV ROC AUC:")
print(results_xgb.head(10).to_string(index=False))


XGBoost with default settings, 5-fold CV ROC AUC: mean 0.9580, std 0.0006
XGBoost grid search, combination 1/96: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8} -> 5-fold CV ROC AUC mean 0.9514, std 0.0004, time 13s
XGBoost grid search, combination 2/96: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 1.0} -> 5-fold CV ROC AUC mean 0.9514, std 0.0004, time 13s
XGBoost grid search, combination 3/96: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 400, 'subsample': 0.8} -> 5-fold CV ROC AUC mean 0.9540, std 0.0005, time 21s
XGBoost grid search, combination 4/96: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 400, 'subsample': 1.0} -> 5-fold CV ROC AUC mean 0.9540, std 0.0005, time 23s
XGBoost grid search, combination 5/96: {'c

In [ ]:
results_xgb = results

In [ ]:
# TRAIN LIGHTGBM CLASSIFIER

from lightgbm import LGBMClassifier

# Train a default model as baseline
scores = cross_val_score(LGBMClassifier(verbose=-1), X, y, cv=cv, scoring='roc_auc')
print(f"LightGBM (default settings), 5-fold CV ROC AUC: mean {scores.mean():.4f}, std {scores.std():.4f}")

# Parameter grid to grid search through
param_grid = {
    'n_estimators': [200, 400],
    'learning_rate': [0.03, 0.1],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [20, 50],
    'reg_lambda': [0, 1, 10],
}

results_lgbm = []

# Define all hyperparameter combinations
combos = list(ParameterGrid(param_grid))

# For each combination...
for i, params in enumerate(combos, 1):
    start = time.time()

    # Train the model using 5-fold CV, scored by ROC AUC
    scores = cross_val_score(LGBMClassifier(**params, verbose=-1), X, y, cv=cv, scoring='roc_auc')

    # Save results (mean + std)
    results_lgbm.append({**params, 'mean_auc': scores.mean(), 'std_auc': scores.std()})
    print(f"LightGBM grid search, combination {i}/{len(combos)}: {params} -> 5-fold CV ROC AUC mean {scores.mean():.4f}, std {scores.std():.4f}, time {time.time() - start:.0f}s")

# Save results to a df & print top 10 
results_lgbm = pd.DataFrame(results_lgbm).sort_values('mean_auc', ascending=False)
print("\nLightGBM grid search, top 10 combinations by 5-fold CV ROC AUC:")
print(results_lgbm.head(10).to_string(index=False))

LightGBM (default settings), 5-fold CV ROC AUC: mean 0.9577, std 0.0006


In [ ]:
# TRAIN LOGISTIC REGRESSION

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# Train a default model as baseline
model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression (default settings), 5-fold CV ROC AUC: mean {scores.mean():.4f}, std {scores.std():.4f}")


# Parameter grid to grid search through
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'class_weight': [None, 'balanced'],
}

results_logreg = []

# Define all hyperparameter combinations
combos = list(ParameterGrid(param_grid))

# For each combination...
for i, params in enumerate(combos, 1):
    start = time.time()

    # Train the model using 5-fold CV, scored by ROC AUC
    model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000, solver='lbfgs', **params))
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')

    # Save results (mean + std)    
    results_logreg.append({**params, 'mean_auc': scores.mean(), 'std_auc': scores.std()})
    print(f"Logistic Regression grid search, combination {i}/{len(combos)}: {params} -> 5-fold CV ROC AUC mean {scores.mean():.4f}, std {scores.std():.4f}, time {time.time() - start:.0f}s")

# Save results to a df & print top 10 
results_logreg = pd.DataFrame(results_logreg).sort_values('mean_auc', ascending=False)
print("\nLogistic Regression grid search, all combinations by 5-fold CV ROC AUC:")
print(results_logreg.to_string(index=False))

Logistic Regression (default settings), 5-fold CV ROC AUC: mean 0.9407, std 0.0005


In [ ]:
# FIND BEST MODEL & RETRAIN ON COMPLETE DATA

# One entry per model type (3 in total). Each entry holds:
## The grid search results table
## A function that builds a model of that type from a given set of hyperparameters.
## Used further down to find the single best combination across all tested models and rebuild it.

candidates = {
    'XGBoost': (results_xgb, lambda p: XGBClassifier(**p)),
    'LightGBM': (results_lgbm, lambda p: LGBMClassifier(**p, verbose=-1)),
    'Logistic Regression': (results_logreg, lambda p: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000, **p))),
}

# Find the model tyoe with the highest score
best_name = max(candidates, key=lambda name: candidates[name][0]['mean_auc'].max())

# Fetch the results table and the model-building function for the winning type
best_results, build_model = candidates[best_name]

# Take the top row (best result) of the winning type's results table
best_params = best_results.iloc[:1].to_dict('records')[0]

# Remove the score columns
best_auc = best_params.pop('mean_auc')
best_params.pop('std_auc')

# Build a new, untrained model of the winning type with the best hyperparameters
final_model = build_model(best_params)

# Train it on ALL training data (not just 4/5 as in cross-validation)
final_model.fit(X, y)
print(f"Final model: {best_name}, params {best_params}, 5-fold CV ROC AUC {best_auc:.4f}")

In [ ]:
# PREDICT FOR TEST SET

# Predict the probability of being satisfied for each test row, using only the same feature columns the model was trained on (excludes id)
submission = pd.DataFrame({
    'id': test_set['id'],
    'satisfaction': final_model.predict_proba(test_set[X.columns])[:, 1],
})

# Save the submission file
submission.to_csv('../submissions/submission.csv', index=False)
print(submission.head())